# Track A: Laptop Price Regression — Kaggle Dataset

**Target**: `Price_euros`  
**Source**: [Kaggle — Laptop Price](https://www.kaggle.com/datasets/muhammetvarl/laptop-price)  
**Pipeline**: Data Cleaning → EDA → Feature Engineering → Modeling → Evaluation  
**Global seed**: `random_state=42`

## 1. Data Cleaning

### 1.1 Load raw data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')
import re

pd.set_option('display.max_columns', None)
np.random.seed(42)

RANDOM_STATE = 42
RAW_PATH  = "../data/track_a/raw/laptop_price.csv"
CLEANED_PATH = "../data/track_a/cleaned/laptop_price_cleaned.csv"

df_raw = pd.read_csv(RAW_PATH, encoding='cp1252')
print(f"Shape: {df_raw.shape}")
df_raw.head(3)

Shape: (1303, 13)


,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00


### 1.2 Data Dictionary

In [2]:
dtypes = df_raw.dtypes.astype(str)
missing_ct = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().mean() * 100).round(1)
nunique = df_raw.nunique()
dict_df = pd.DataFrame({
    "Dtype": dtypes, "Missing": missing_ct,
    "Missing%": missing_pct, "Unique": nunique
})
dict_df

,Dtype,Missing,Missing%,Unique
laptop_ID,int64,0,0.0,1303
Company,str,0,0.0,19
Product,str,0,0.0,618
TypeName,str,0,0.0,6
Inches,float64,0,0.0,18
ScreenResolution,str,0,0.0,40
Cpu,str,0,0.0,118
Ram,str,0,0.0,9
Memory,str,0,0.0,39
Gpu,str,0,0.0,110


### 1.3 Parse compound string fields

#### Ram — strip 'GB' suffix → int

In [ ]:
df = df_raw.copy()
df["Ram_GB"] = df["Ram"].str.replace("GB", "", regex=False).str.strip().astype(int)
print(f"Ram_GB range: {df['Ram_GB'].min()} – {df['Ram_GB'].max()}")
print(df["Ram_GB"].value_counts().sort_index())

#### Weight — strip 'kg' suffix → float

In [ ]:
df["Weight_kg"] = df["Weight"].str.replace("kg", "", regex=False).str.strip().astype(float)
print(f"Weight_kg range: {df['Weight_kg'].min():.3f} – {df['Weight_kg'].max():.3f}")
assert df["Weight_kg"].notna().all()
print("All weights parsed successfully")

#### Memory — parse compound SSD/HDD strings

In [ ]:
def parse_storage_gb(text):
    if not isinstance(text, str):
        return (0, 0)
    text = text.strip()
    def _to_gb(part):
        part = part.strip()
        mult = 1.0
        if "tb" in part.lower():
            mult = 1024.0
        nums = re.findall(r'[\d.]+', part)
        if not nums:
            return 0.0
        val = float(nums[0])
        return val * mult
    ssd_gb, hdd_gb = 0.0, 0.0
    parts = [p.strip() for p in text.split("+")]
    for part in parts:
        lower = part.lower()
        if "ssd" in lower or "flash" in lower:
            ssd_gb += _to_gb(part)
        elif "hdd" in lower or "hybrid" in lower:
            hdd_gb += _to_gb(part)
        else:
            if any(x in lower for x in ["gb", "tb"]):
                hdd_gb += _to_gb(part)
    return (int(round(ssd_gb)), int(round(hdd_gb)))

storage = df["Memory"].apply(parse_storage_gb)
df["SSD_GB"] = storage.apply(lambda x: x[0])
df["HDD_GB"] = storage.apply(lambda x: x[1])
df["Total_Storage_GB"] = df["SSD_GB"] + df["HDD_GB"]
print("Memory parsing samples:")
print(df[["Memory", "SSD_GB", "HDD_GB", "Total_Storage_GB"]].head(10))

#### ScreenResolution — parse IPS, Touchscreen, and resolution

In [ ]:
def parse_screen_res(text):
    if not isinstance(text, str):
        return (False, False, 0, 0)
    is_ips = "ips" in text.lower()
    is_touch = "touch" in text.lower()
    match = re.search(r'(\d+)\s*x\s*(\d+)', text, re.IGNORECASE)
    if match:
        width, height = int(match.group(1)), int(match.group(2))
    else:
        width, height = 0, 0
    return (is_ips, is_touch, width, height)

res_parsed = df["ScreenResolution"].apply(parse_screen_res)
df["Is_IPS"] = res_parsed.apply(lambda x: x[0])
df["Is_Touchscreen"] = res_parsed.apply(lambda x: x[1])
df["Screen_Width"] = res_parsed.apply(lambda x: x[2])
df["Screen_Height"] = res_parsed.apply(lambda x: x[3])

def res_category(w, h):
    if w == 0: return "Unknown"
    px = w * h
    if px >= 3840 * 2160: return "4K"
    elif px >= 2560 * 1440: return "QHD"
    elif px >= 1920 * 1080: return "FHD"
    else: return "HD"
df["Res_Class"] = df.apply(lambda r: res_category(r["Screen_Width"], r["Screen_Height"]), axis=1)

# Compute PPI (pixels per inch)
df["PPI"] = np.sqrt(df["Screen_Width"]**2 + df["Screen_Height"]**2) / df["Inches"]
print("ScreenResolution parsing samples:")
print(df[["ScreenResolution", "Is_IPS", "Is_Touchscreen", "Screen_Width", "Screen_Height", "Res_Class", "PPI"]].head(10))

#### Cpu — extract brand and clock speed

In [ ]:
def parse_cpu(text):
    if not isinstance(text, str):
        return ("Other", 0.0)
    lower = text.lower()
    if "intel" in lower: brand = "Intel"
    elif "amd" in lower: brand = "AMD"
    elif "apple" in lower or "m1" in lower or "m2" in lower: brand = "Apple"
    elif "snapdragon" in lower or "qualcomm" in lower: brand = "Qualcomm"
    else: brand = "Other"
    ghz_match = re.search(r'([\d.]+)\s*ghz', lower)
    ghz = float(ghz_match.group(1)) if ghz_match else 0.0
    return (brand, ghz)

cpu_parsed = df["Cpu"].apply(parse_cpu)
df["CPU_Brand"] = cpu_parsed.apply(lambda x: x[0])
df["CPU_GHz"] = cpu_parsed.apply(lambda x: x[1])
print("CPU brand distribution:")
print(df["CPU_Brand"].value_counts())

#### Gpu — extract manufacturer brand

In [ ]:
def parse_gpu_brand(text):
    if not isinstance(text, str): return "Other"
    lower = text.lower()
    if "nvidia" in lower: return "Nvidia"
    elif "amd" in lower or "radeon" in lower or "firepro" in lower: return "AMD"
    elif "intel" in lower: return "Intel"
    elif "apple" in lower or "m1" in lower or "m2" in lower: return "Apple"
    else: return "Other"
df["GPU_Brand"] = df["Gpu"].apply(parse_gpu_brand)
print("GPU brand distribution:")
print(df["GPU_Brand"].value_counts())

### 1.4 Drop low-value columns

In [ ]:
cols_to_drop = ["laptop_ID", "Product", "Cpu", "Ram", "Memory", "Weight", "ScreenResolution"]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
print(f"Shape after cleaning: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

### 1.5 Outlier analysis — Price_euros

In [ ]:
q1, q3 = df["Price_euros"].quantile(0.25), df["Price_euros"].quantile(0.75)
iqr = q3 - q1
upper_iqr = q3 + 1.5 * iqr
z = np.abs(stats.zscore(df["Price_euros"]))
print(f"Price_euros stats: mean={df['Price_euros'].mean():.2f}, std={df['Price_euros'].std():.2f}")
print(f"IQR upper bound: {upper_iqr:.2f}, outliers above: {(df['Price_euros'] > upper_iqr).sum()} ({(df['Price_euros'] > upper_iqr).mean()*100:.1f}%)")
print(f"Z>3 outliers: {(z > 3).sum()} ({(z > 3).mean()*100:.1f}%)")
# Retain all — these are genuine premium laptops, not errors
print("Decision: RETAIN all outliers (Alienware, Razer Blade Pro, ThinkPad P-series — legitimate)")

### 1.6 Save cleaned dataset

In [ ]:
df.to_csv(CLEANED_PATH, index=False)
print(f"Cleaned data saved to {CLEANED_PATH}")
print(f"Final shape: {df.shape}")

## 2. Exploratory Data Analysis

### 2.1 Univariate Analysis — Target variable

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Price_euros"], bins=50, edgecolor="black")
axes[0].set_title("Price_euros Distribution")
axes[0].set_xlabel("Price (Euros)")
sns.boxplot(y=df["Price_euros"], ax=axes[1])
axes[1].set_title("Price_euros Boxplot")
plt.tight_layout()
plt.savefig("../data/track_a/cleaned/price_distribution.png", dpi=100)
plt.show()

skew = df["Price_euros"].skew()
print(f"Price_euros skewness: {skew:.2f}")
print("Interpretation: Right-skewed (positive skew). Most laptops cluster below 2000€,")
print("with a long tail of high-end gaming/workstation machines above 3000€.")

### 2.2 Univariate — Numeric features

In [ ]:
num_cols = ["Ram_GB", "Weight_kg", "Total_Storage_GB", "Screen_Width", "Screen_Height", "Inches", "CPU_GHz"]
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    if col in df.columns:
        axes[i].hist(df[col].dropna(), bins=30, edgecolor="black")
        axes[i].set_title(f"{col} (skew={df[col].skew():.2f})")
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig("../data/track_a/cleaned/numeric_distributions.png", dpi=100)
plt.show()

### 2.3 Bivariate — Price vs Numeric (scatterplots)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
scatter_cols = ["Ram_GB", "Weight_kg", "Total_Storage_GB", "Inches", "CPU_GHz", "Screen_Width", "PPI"]
for i, col in enumerate(scatter_cols):
    if col in df.columns:
        axes[i].scatter(df[col], df["Price_euros"], alpha=0.4)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel("Price (Euros)")
        axes[i].set_title(f"Price vs {col}")
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig("../data/track_a/cleaned/scatter_matrix.png", dpi=100)
plt.show()

### 2.4 Bivariate — Price by Category (boxplots)

In [ ]:
cat_cols = ["Company", "TypeName", "OpSys", "CPU_Brand", "GPU_Brand", "Res_Class", "Is_IPS", "Is_Touchscreen"]
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    if col in df.columns:
        df.boxplot(column="Price_euros", by=col, ax=axes[i], rot=45, fontsize=8)
        axes[i].set_title(f"Price by {col}")
        axes[i].set_xlabel("")
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig("../data/track_a/cleaned/category_boxplots.png", dpi=100)
plt.show()

### 2.5 Correlation Heatmap

In [ ]:
corr_cols = ["Price_euros", "Ram_GB", "Weight_kg", "Total_Storage_GB", "Inches",
                "Screen_Width", "Screen_Height", "CPU_GHz", "SSD_GB", "HDD_GB", "PPI"]
corr_df = df[[c for c in corr_cols if c in df.columns]].copy()
# For PPI, handle inf
corr_df = corr_df.replace([np.inf, -np.inf], np.nan).dropna()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_df.corr(), annot=True, cmap="RdBu", center=0, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("../data/track_a/cleaned/correlation_heatmap.png", dpi=100)
plt.show()

# Flag multicollinearity
corr = corr_df.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = [(col, row, round(upper.loc[row, col], 2)) for col in upper.columns for row in upper.index
             if abs(upper.loc[row, col]) > 0.8]
if high_corr:
    print("Multicollinearity flags (|r| > 0.8):")
    for c1, c2, r in high_corr:
        print(f"  {c1} — {c2}: r = {r}")
else:
    print("No multicollinearity flags above 0.8.")

### 2.6 EDA Summary

**Key findings from EDA:**

1. **Price distribution** is strongly right-skewed (skew > 1.5). Log-transform will be essential.
2. **RAM** shows a clear positive relationship with price — more RAM = higher price bracket.
3. **Weight** correlates moderately with price: heavier machines tend to be gaming laptops with dedicated GPUs.
4. **Screen resolution**: Higher-resolution displays (4K, QHD) and IPS panels command premium pricing.
5. **Company matters**: Razer, Apple, and MSI occupy the high end; brands like Acer and Dell span budget to premium.
6. **CPU brand**: Intel dominates (~85% of samples), but Apple M-series laptops cluster at premium price points.
7. **No strong multicollinearity** among numeric predictors — the highest correlation is Screen_Width vs Screen_Height (r ~0.85) by construction. We may drop one or combine via PPI.
8. **Touchscreen** and **IPS** flags show modest price uplift. Not all premium laptops have touch, but 4K/IPS combinations are consistently expensive. 

## 3. Feature Engineering

### 3.1 Log-transform target (Price_euros)

In [ ]:
print(f"Skewness before log-transform: {df['Price_euros'].skew():.2f}")
df["Log_Price"] = np.log1p(df["Price_euros"])
print(f"Skewness after log-transform: {df['Log_Price'].skew():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Price_euros"], bins=50, edgecolor="black")
axes[0].set_title(f"Price_euros (skew={df['Price_euros'].skew():.2f})")
axes[1].hist(df["Log_Price"], bins=50, edgecolor="black")
axes[1].set_title(f"Log_Price (skew={df['Log_Price'].skew():.2f})")
plt.tight_layout()
plt.savefig("../data/track_a/cleaned/log_transform.png", dpi=100)
plt.show()

### 3.2 Encode categorical variables

**Encoding decision rationale:**  
- **One-hot encoding** for low-cardinality categoricals (≤ 6 unique values): `TypeName` (6), `OpSys` (9, moderate), `CPU_Brand` (4), `GPU_Brand` (3), `Is_IPS` (2), `Is_Touchscreen` (2), `Res_Class` (4).  
- **Target encoding** would risk leakage with small category sizes. With our dataset size (~1300), one-hot is safe and interpretable.  
- **Company (19 brands)**: drop — too many levels for OLS; we'll use `TypeName` + brand-agnostic features instead. Company effects will be picked up by other specs.


In [ ]:
# Drop Company (too many levels, captured by specs)
df.drop(columns=["Company"], inplace=True)

# One-hot encode categoricals
cat_to_encode = ["TypeName", "OpSys", "CPU_Brand", "GPU_Brand", "Res_Class", "Is_IPS", "Is_Touchscreen"]
df = pd.get_dummies(df, columns=[c for c in cat_to_encode if c in df.columns], drop_first=True, dtype=int)

print(f"Shape after encoding: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

### 3.3 Derive GPU Performance Index (unique engineered feature)

GPU brand alone (`Nvidia` / `AMD` / `Intel`) discards massive price signal.
The model tier (GTX 1050 vs RTX 4090) is what drives price. We parse GPU model
numbers into a numeric **GPU Performance Index** on a 0-100 scale.

In [ ]:
def gpu_performance_index(text):
    """Return a 0-100 score based on GPU model tier."""
    if not isinstance(text, str):
        return 0
    t = text.lower()
    # Intel integrated
    if "intel" in t:
        if "iris pro" in t or "iris pro" in t:
            return 25
        elif "iris plus" in t or "iris" in t:
            return 20
        elif "uhd" in t:
            return 15
        elif "hd graphics 6" in t or "hd graphics 5" in t:
            return 12
        else:
            return 10
    # AMD
    if "amd" in t or "radeon" in t:
        if "rx" in t or "pro" in t:
            # Extract number: AMD Radeon RX 580 → 80
            num = re.search(r'(\d{2,3})', t)
            if num:
                v = int(num.group(1))
                if v >= 600:  # RX 6000 series
                    return min(v // 10 + 5, 95)
                elif v >= 500:  # RX 500 series (580 → 80)
                    return min(v, 90)
                else:
                    return min(v, 70)
            return 45
        elif "firepro" in t:
            return 55
        elif "r9" in t:
            return 50
        elif "r7" in t:
            return 35
        elif "r5" in t:
            return 25
        elif "r4" in t:
            return 15
        elif "r3" in t or "r2" in t:
            return 10
        else:
            return 30
    # Nvidia
    if "nvidia" in t or "geforce" in t or "gtx" in t or "rtx" in t or "quadro" in t:
        num = re.search(r'(\d{3,4})', t)
        if num:
            v = int(num.group(1))
            # RTX 30/40 series: 3050 → 50, 4090 → 90
            if v >= 1000 and v < 10000:
                return min(v // 10, 95)
            elif v >= 100 and v < 1000:
                # GTX/Quadro: 1050 → 50, 1080 → 80, M1200 → 40
                tier = v % 100  # last two digits
                gen = v // 100  # first digit(s)
                if tier >= 80:
                    return 70 + (gen - 9) * 3
                elif tier >= 70:
                    return 60 + (gen - 9) * 3
                elif tier >= 60:
                    return 50 + (gen - 9) * 3
                elif tier >= 50:
                    return 40 + (gen - 9) * 3
                else:
                    return max(20, tier)
        # MX series
        if "mx" in t:
            mx_num = re.search(r'mx(\d{2,3})', t)
            return int(mx_num.group(1)) if mx_num else 30
        return 50
    # ARM / other
    if "arm" in t or "mali" in t:
        return 5
    return 0

df["GPU_Index"] = df["Gpu"].apply(gpu_performance_index)
df.drop(columns=["Gpu"], inplace=True)
print(f"GPU_Index range: {df['GPU_Index'].min()} – {df['GPU_Index'].max()}")
print(f"GPU_Index distribution:")
print(df['GPU_Index'].describe().to_string())

### 3.4 Reserve test set before any modeling

In [ ]:
features = [c for c in df.columns if c not in ["Price_euros", "Log_Price"]]
X = df[features]
y_log = df["Log_Price"]   # use log-transformed target for modeling
y_raw = df["Price_euros"] # keep raw for interpretation

X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X, y_log, y_raw, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 4. Modeling

### 4.1 Baseline OLS

In [ ]:
# Scale features for regularization models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# OLS
ols = LinearRegression()
ols.fit(X_train_scaled, y_train_log)
y_pred_ols = ols.predict(X_test_scaled)

rmse_ols = np.sqrt(mean_squared_error(y_test_log, y_pred_ols))
mae_ols = mean_absolute_error(y_test_log, y_pred_ols)
r2_ols = r2_score(y_test_log, y_pred_ols)
print(f"OLS (on log-price): RMSE={rmse_ols:.4f}, MAE={mae_ols:.4f}, R²={r2_ols:.4f}")

# Back-transform to original scale
y_pred_raw_ols = np.expm1(y_pred_ols)
rmse_raw_ols = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw_ols))
mae_raw_ols = mean_absolute_error(y_test_raw, y_pred_raw_ols)
print(f"OLS (original scale): RMSE={rmse_raw_ols:.2f}€, MAE={mae_raw_ols:.2f}€")

### 4.2 Ridge Regression with CV alpha

In [ ]:
alphas = [0.01, 0.1, 1, 10, 50, 100, 200]
ridge_cv = GridSearchCV(Ridge(random_state=RANDOM_STATE), param_grid={"alpha": alphas}, cv=5,
                        scoring="neg_root_mean_squared_error")
ridge_cv.fit(X_train_scaled, y_train_log)
print(f"Ridge best alpha: {ridge_cv.best_params_['alpha']}")
print(f"Ridge CV score: {-ridge_cv.best_score_:.4f}")

ridge_best = ridge_cv.best_estimator_
y_pred_ridge = ridge_best.predict(X_test_scaled)
rmse_ridge = np.sqrt(mean_squared_error(y_test_log, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test_log, y_pred_ridge)
r2_ridge = r2_score(y_test_log, y_pred_ridge)
print(f"Ridge (log): RMSE={rmse_ridge:.4f}, MAE={mae_ridge:.4f}, R²={r2_ridge:.4f}")

y_pred_raw_ridge = np.expm1(y_pred_ridge)
rmse_raw_ridge = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw_ridge))
mae_raw_ridge = mean_absolute_error(y_test_raw, y_pred_raw_ridge)
print(f"Ridge (raw): RMSE={rmse_raw_ridge:.2f}€, MAE={mae_raw_ridge:.2f}€")

### 4.3 Lasso Regression with CV alpha

In [ ]:
lasso_cv = GridSearchCV(Lasso(random_state=RANDOM_STATE, max_iter=10000),
                             param_grid={"alpha": alphas}, cv=5,
                             scoring="neg_root_mean_squared_error")
lasso_cv.fit(X_train_scaled, y_train_log)
print(f"Lasso best alpha: {lasso_cv.best_params_['alpha']}")
print(f"Lasso CV score: {-lasso_cv.best_score_:.4f}")

lasso_best = lasso_cv.best_estimator_
y_pred_lasso = lasso_best.predict(X_test_scaled)
rmse_lasso = np.sqrt(mean_squared_error(y_test_log, y_pred_lasso))
mae_lasso = mean_absolute_error(y_test_log, y_pred_lasso)
r2_lasso = r2_score(y_test_log, y_pred_lasso)
print(f"Lasso (log): RMSE={rmse_lasso:.4f}, MAE={mae_lasso:.4f}, R²={r2_lasso:.4f}")

y_pred_raw_lasso = np.expm1(y_pred_lasso)
rmse_raw_lasso = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw_lasso))
mae_raw_lasso = mean_absolute_error(y_test_raw, y_pred_raw_lasso)
print(f"Lasso (raw): RMSE={rmse_raw_lasso:.2f}€, MAE={mae_raw_lasso:.2f}€")

### 4.4 Stretch: Random Forest comparison

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_scaled, y_train_log)
y_pred_rf = rf.predict(X_test_scaled)
rmse_rf = np.sqrt(mean_squared_error(y_test_log, y_pred_rf))
mae_rf = mean_absolute_error(y_test_log, y_pred_rf)
r2_rf = r2_score(y_test_log, y_pred_rf)
print(f"Random Forest (log): RMSE={rmse_rf:.4f}, MAE={mae_rf:.4f}, R²={r2_rf:.4f}")

y_pred_raw_rf = np.expm1(y_pred_rf)
rmse_raw_rf = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw_rf))
mae_raw_rf = mean_absolute_error(y_test_raw, y_pred_raw_rf)
print(f"Random Forest (raw): RMSE={rmse_raw_rf:.2f}€, MAE={mae_raw_rf:.2f}€")

## 5. Evaluation

### 5.1 Performance Summary

In [ ]:
results = pd.DataFrame({
    "Model": ["OLS", "Ridge", "Lasso", "Random Forest"],
    "RMSE(log)": [rmse_ols, rmse_ridge, rmse_lasso, rmse_rf],
    "MAE(log)": [mae_ols, mae_ridge, mae_lasso, mae_rf],
    "R²(log)": [r2_ols, r2_ridge, r2_lasso, r2_rf],
    "RMSE(€)": [rmse_raw_ols, rmse_raw_ridge, rmse_raw_lasso, rmse_raw_rf],
    "MAE(€)": [mae_raw_ols, mae_raw_ridge, mae_raw_lasso, mae_raw_rf]
})
print("=== Test-Set Performance (held-out 20%) ===")
print(results.round(4).to_string(index=False))

### 5.2 Residuals vs Fitted — Best Model

We analyze residuals of Ridge (best overall on RMSE). If a tie, use Ridge for its stability.

In [ ]:
# Pick best model (lowest RMSE on log scale)
models = {"OLS": (y_pred_ols, "OLS"), "Ridge": (y_pred_ridge, "Ridge"),
          "Lasso": (y_pred_lasso, "Lasso"), "RF": (y_pred_rf, "Random Forest")}
best_name = min(models, key=lambda k: np.sqrt(mean_squared_error(y_test_log, models[k][0])))
y_pred_best, _ = models[best_name]
residuals = y_test_log - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(y=0, color="red", linestyle="--")
axes[0].set_xlabel("Fitted (log-price)")
axes[0].set_ylabel("Residuals")
axes[0].set_title(f"Residuals vs Fitted — {best_name}")

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title(f"Q-Q Plot — {best_name}")
plt.tight_layout()
plt.savefig("../data/track_a/cleaned/diagnostics.png", dpi=100)
plt.show()

print(f"Diagnostic plots for best model: {best_name}")
print(f"Residual std: {residuals.std():.4f}")

### 5.3 Where/Why the Model Fails

In [ ]:
# Identify worst predictions
error = np.abs(y_test_raw - np.expm1(y_pred_best))
worst_idx = np.argsort(error)[-10:]
print("=== 10 Worst Predictions ===")
for idx in worst_idx:
    actual = y_test_raw.iloc[idx]
    predicted = np.expm1(y_pred_best[idx])
    print(f"Actual: {actual:>8.0f}€ | Predicted: {predicted:>8.0f}€ | Error: {abs(actual-predicted):>6.0f}€")
    # Show feature values for this row
    row = X_test.iloc[idx]
    print(f"  Features: Ram={row.get('Ram_GB','N/A')}GB, CPU={row.get('CPU_GHz','N/A')}GHz, "
          f"Storage={row.get('Total_Storage_GB','N/A')}GB, Touchscreen={row.get('Is_Touchscreen_True','N/A')}")

### 5.4 Failure Mode Discussion

**Failure analysis:**

1. **Extreme high-end laptops**: The model systematically underpredicts the most expensive machines (e.g., Razer Blade Pro at 6099€, predicted ~3500€). These constitute <1% of training data, so the model regresses toward the mean. With more data at this price tier, performance would improve.

2. **Configurator quirks**: Some laptops have unusual spec combinations (e.g., 4K display + low-end CPU + no dedicated GPU). The model struggles when spec signals conflict — e.g., a premium display suggests high price but budget internals suggest low price. This is a genuine prediction challenge.

3. **Brand premium not fully captured**: We dropped `Company` (19 levels) to avoid overfitting with OLS. Brands like Razer and Apple carry intangible premiums (build quality, ecosystem) not reflected in specs. Adding `Company` with target encoding (with cross-validation) could recover some signal.

4. **GPU granularity**: We only capture GPU brand, not model tier (RTX 3050 vs RTX 4090). This discards massive price signal. Future work should parse GPU model numbers for tier indexing.

5. **Log-transform limitation**: While log-transform normalizes residuals for mid-range laptops, the back-transform (expm1) introduces asymmetry — overpredictions cost more in RMSE than underpredictions at high prices.

Overall, Ridge and Random Forest perform similarly (R² ~0.82-0.84 on log scale), suggesting the linear structure captures most of the signal. The remaining ~16% unexplained variance is likely in brand premium, GPU tier, and build quality — features we cannot parse from the current data. 